# Reach Diffusion U-Net Training

This notebook trains a conditional diffusion policy for `reach_data_1`.

- Input context: processed `obs` from `reach_data_1/obs`
- Output target: 16-step `reach_imitation_action_chunk`
- Gripper preprocessing: average the last two finger joints into a single scalar
- Observation preprocessing: drop the final `last_action` part from the raw observation


In [1]:
import math
import copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [2]:
class Config:
    data_root = Path("../../datas/reach_lift_data_1")
    obs_dir = data_root / "obs"
    act_dir = data_root / "actions"

    chunk_size = 16
    batch_size = 256
    num_epochs = 200
    learning_rate = 1e-4
    weight_decay = 1e-6
    num_workers = 4

    diffusion_steps = 100
    beta_start = 1e-4
    beta_end = 2e-2
    schedule = "cosine"

    base_channels = 128
    time_embed_dim = 128
    obs_embed_dim = 128

    raw_obs_dim = 32
    processed_obs_dim = 31
    raw_action_dim = 9
    processed_action_dim = 8

    use_ema = True
    ema_decay = 0.999

    save_dir = Path("./reach_lift_diffusion_unet_artifacts")
    checkpoint_path = save_dir / "reach_lift_diffusion_unet.pt"
    stats_path = save_dir / "reach_lift_diffusion_stats.npz"

config = Config()
config.save_dir.mkdir(parents=True, exist_ok=True)
config


In [3]:
def preprocess_reach_obs(raw_obs: np.ndarray, raw_action: np.ndarray) -> np.ndarray:
    """Raw reach obs layout: [joint_pos(9), joint_vel(9), pose_command(7), last_action(7)]."""
    if raw_obs.shape[0] != raw_action.shape[0]:
        n = min(raw_obs.shape[0], raw_action.shape[0])
        raw_obs = raw_obs[:n]
        raw_action = raw_action[:n]

    joint_pos = raw_obs[:, 0:9]
    joint_vel = raw_obs[:, 9:18]
    pose_command = raw_obs[:, 18:25]

    joint_pos_processed = np.concatenate([joint_pos[:, :7], joint_pos[:, 7:9].mean(axis=1, keepdims=True)], axis=1)
    joint_vel_processed = np.concatenate([joint_vel[:, :7], joint_vel[:, 7:9].mean(axis=1, keepdims=True)], axis=1)

    # Override last_action with processed actions from dataset (gripper mean)
    last_action = preprocess_reach_action(raw_action)

    return np.concatenate([joint_pos_processed, joint_vel_processed, pose_command, last_action], axis=1).astype(np.float32)


def preprocess_reach_action(raw_action: np.ndarray) -> np.ndarray:
    """Raw saved action layout: [7 non-gripper joints, left_finger, right_finger]."""
    gripper_mean = raw_action[:, 7:9].mean(axis=1, keepdims=True)
    return np.concatenate([raw_action[:, :7], gripper_mean], axis=1).astype(np.float32)


def compute_window_weights(n: int, chunk_size: int) -> np.ndarray:
    t = np.arange(n)
    start_min = np.maximum(0, t - chunk_size + 1)
    start_max = np.minimum(t, n - chunk_size)
    w = start_max - start_min + 1
    return np.maximum(w, 0).astype(np.float64)


def build_episode_index(obs_dir: Path, act_dir: Path, chunk_size: int):
    obs_paths = sorted(obs_dir.glob("obs_ep_*.npy"))
    episodes = []
    for obs_path in tqdm(obs_paths, desc="Indexing episodes"):
        act_path = act_dir / obs_path.name.replace("obs_", "act_")
        if not act_path.exists():
            continue
        obs = np.load(obs_path, mmap_mode="r")
        act = np.load(act_path, mmap_mode="r")
        n = min(len(obs), len(act))
        if n < chunk_size:
            continue
        episodes.append((obs_path, act_path, n))
    return episodes


def compute_stats(episodes, chunk_size: int):
    obs_sum = np.zeros((config.processed_obs_dim,), dtype=np.float64)
    obs_sq = np.zeros((config.processed_obs_dim,), dtype=np.float64)
    obs_count = 0

    act_sum = np.zeros((config.processed_action_dim,), dtype=np.float64)
    act_sq = np.zeros((config.processed_action_dim,), dtype=np.float64)
    act_weight_sum = 0.0

    for obs_path, act_path, n in tqdm(episodes, desc="Computing stats"):
        raw_obs = np.load(obs_path)
        raw_act = np.load(act_path)
        n = min(len(raw_obs), len(raw_act))
        if n < chunk_size:
            continue
        raw_obs = raw_obs[:n]
        raw_act = raw_act[:n]

        proc_obs = preprocess_reach_obs(raw_obs, raw_act)
        proc_act = preprocess_reach_action(raw_act)

        valid_len = n - chunk_size + 1
        proc_obs = proc_obs[:valid_len]

        obs_sum += proc_obs.sum(axis=0)
        obs_sq += (proc_obs ** 2).sum(axis=0)
        obs_count += valid_len

        weights = compute_window_weights(n, chunk_size)[:, None]
        act_sum += (proc_act * weights).sum(axis=0)
        act_sq += ((proc_act ** 2) * weights).sum(axis=0)
        act_weight_sum += float(weights.sum())

    obs_mean = obs_sum / max(obs_count, 1)
    obs_var = obs_sq / max(obs_count, 1) - obs_mean ** 2
    obs_std = np.sqrt(np.maximum(obs_var, 1e-12)) + 1e-6

    act_mean = act_sum / max(act_weight_sum, 1)
    act_var = act_sq / max(act_weight_sum, 1) - act_mean ** 2
    act_std = np.sqrt(np.maximum(act_var, 1e-12)) + 1e-6

    return (
        obs_mean.astype(np.float32),
        obs_std.astype(np.float32),
        act_mean.astype(np.float32),
        act_std.astype(np.float32),
    )


episodes = build_episode_index(config.obs_dir, config.act_dir, config.chunk_size)

rng = np.random.default_rng(42)
perm = rng.permutation(len(episodes))
train_count = int(len(episodes) * 0.95)
train_episodes = [episodes[i] for i in perm[:train_count]]
val_episodes = [episodes[i] for i in perm[train_count:]]

obs_mean, obs_std, act_mean, act_std = compute_stats(train_episodes, config.chunk_size)

np.savez(
    config.stats_path,
    obs_mean=obs_mean,
    obs_std=obs_std,
    act_mean=act_mean,
    act_std=act_std,
)

len(episodes), len(train_episodes), len(val_episodes)


Computing stats: 100%|██████████| 13705/13705 [00:21<00:00, 637.76it/s] 


(14427, 13705, 722)

In [4]:
class ReachChunkStreamingDataset(Dataset):
    def __init__(
        self,
        episodes,
        chunk_size: int,
        obs_mean: np.ndarray,
        obs_std: np.ndarray,
        act_mean: np.ndarray,
        act_std: np.ndarray,
    ):
        self.episodes = episodes
        self.chunk_size = chunk_size
        self.obs_mean = obs_mean.astype(np.float32)
        self.obs_std = obs_std.astype(np.float32)
        self.act_mean = act_mean.astype(np.float32)
        self.act_std = act_std.astype(np.float32)

        self.window_counts = np.asarray([n - chunk_size + 1 for _, _, n in episodes], dtype=np.int64)
        self.cum_windows = np.cumsum(self.window_counts)
        self.total_windows = int(self.cum_windows[-1]) if len(self.cum_windows) > 0 else 0

        self._cache_ep = None
        self._cache_obs = None
        self._cache_act = None

    def __len__(self):
        return self.total_windows

    def _load_episode(self, ep_idx: int):
        if self._cache_ep == ep_idx:
            return self._cache_obs, self._cache_act
        obs_path, act_path, _ = self.episodes[ep_idx]
        obs = np.load(obs_path, mmap_mode="r")
        act = np.load(act_path, mmap_mode="r")
        self._cache_ep = ep_idx
        self._cache_obs = obs
        self._cache_act = act
        return obs, act

    def __getitem__(self, idx: int):
        if idx < 0 or idx >= self.total_windows:
            raise IndexError(f"Index {idx} out of range for dataset of size {self.total_windows}")

        ep_idx = int(np.searchsorted(self.cum_windows, idx, side="right"))
        prev = 0 if ep_idx == 0 else self.cum_windows[ep_idx - 1]
        local_start = int(idx - prev)

        obs_raw, act_raw = self._load_episode(ep_idx)
        obs_slice = np.asarray(obs_raw[local_start : local_start + 1])
        act_slice = np.asarray(act_raw[local_start : local_start + self.chunk_size])

        obs = preprocess_reach_obs(obs_slice, act_slice[:1])[0]
        act = preprocess_reach_action(act_slice)

        obs = (obs - self.obs_mean) / self.obs_std
        act = (act - self.act_mean) / self.act_std

        return torch.from_numpy(obs.astype(np.float32)), torch.from_numpy(act.astype(np.float32))


train_dataset = ReachChunkStreamingDataset(
    train_episodes,
    config.chunk_size,
    obs_mean,
    obs_std,
    act_mean,
    act_std,
)
val_dataset = ReachChunkStreamingDataset(
    val_episodes,
    config.chunk_size,
    obs_mean,
    obs_std,
    act_mean,
    act_std,
)

dataset = train_dataset

loader_kwargs = {
    "batch_size": config.batch_size,
    "num_workers": config.num_workers,
    "pin_memory": False,
}
if config.num_workers > 0:
    loader_kwargs["persistent_workers"] = True
    loader_kwargs["prefetch_factor"] = 2

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    **loader_kwargs,
)
val_loader = DataLoader(
    val_dataset,
    shuffle=False,
    **loader_kwargs,
)

len(train_dataset), len(val_dataset)


(19614042, 1038829)

In [5]:
class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        scale = math.log(10000) / max(half - 1, 1)
        emb = torch.exp(torch.arange(half, device=t.device) * -scale)
        emb = t[:, None] * emb[None, :]
        return torch.cat([emb.sin(), emb.cos()], dim=-1)


class ResBlock1D(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, cond_dim: int):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1)
        self.cond_proj = nn.Linear(cond_dim, out_channels)
        self.skip = nn.Conv1d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x: torch.Tensor, cond: torch.Tensor) -> torch.Tensor:
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.cond_proj(cond).unsqueeze(-1)
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class ConditionalUnet1D(nn.Module):
    def __init__(self, action_dim: int, obs_dim: int, base_channels: int, time_embed_dim: int, obs_embed_dim: int):
        super().__init__()
        self.action_dim = action_dim
        self.time_mlp = nn.Sequential(
            SinusoidalPosEmb(time_embed_dim),
            nn.Linear(time_embed_dim, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, time_embed_dim),
        )
        self.obs_mlp = nn.Sequential(
            nn.Linear(obs_dim, obs_embed_dim),
            nn.SiLU(),
            nn.Linear(obs_embed_dim, obs_embed_dim),
        )
        self.cond_mlp = nn.Sequential(
            nn.Linear(time_embed_dim + obs_embed_dim, base_channels),
            nn.SiLU(),
            nn.Linear(base_channels, base_channels),
        )

        self.input_proj = nn.Conv1d(action_dim, base_channels, kernel_size=3, padding=1)
        self.down1 = ResBlock1D(base_channels, base_channels, base_channels)
        self.downsample1 = nn.Conv1d(base_channels, base_channels, kernel_size=4, stride=2, padding=1)
        self.down2 = ResBlock1D(base_channels, base_channels * 2, base_channels)
        self.downsample2 = nn.Conv1d(base_channels * 2, base_channels * 2, kernel_size=4, stride=2, padding=1)

        self.mid1 = ResBlock1D(base_channels * 2, base_channels * 2, base_channels)
        self.mid2 = ResBlock1D(base_channels * 2, base_channels * 2, base_channels)

        self.upsample1 = nn.ConvTranspose1d(base_channels * 2, base_channels * 2, kernel_size=4, stride=2, padding=1)
        self.up1 = ResBlock1D(base_channels * 4, base_channels, base_channels)
        self.upsample2 = nn.ConvTranspose1d(base_channels, base_channels, kernel_size=4, stride=2, padding=1)
        self.up2 = ResBlock1D(base_channels * 2, base_channels, base_channels)

        self.output_proj = nn.Sequential(
            nn.GroupNorm(8, base_channels),
            nn.SiLU(),
            nn.Conv1d(base_channels, action_dim, kernel_size=3, padding=1),
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor, obs: torch.Tensor) -> torch.Tensor:
        time_emb = self.time_mlp(t)
        obs_emb = self.obs_mlp(obs)
        cond = self.cond_mlp(torch.cat([time_emb, obs_emb], dim=-1))

        x0 = self.input_proj(x)
        d1 = self.down1(x0, cond)
        d2 = self.down2(self.downsample1(d1), cond)
        mid = self.mid2(self.mid1(self.downsample2(d2), cond), cond)

        u1 = self.upsample1(mid)
        u1 = self.up1(torch.cat([u1, d2], dim=1), cond)
        u2 = self.upsample2(u1)
        u2 = self.up2(torch.cat([u2, d1], dim=1), cond)
        return self.output_proj(u2)


def cosine_beta_schedule(num_steps: int, s: float = 0.008) -> torch.Tensor:
    steps = num_steps + 1
    t = torch.linspace(0, num_steps, steps) / num_steps
    alpha_bars = torch.cos(((t + s) / (1 + s)) * math.pi * 0.5) ** 2
    alpha_bars = alpha_bars / alpha_bars[0]
    betas = 1 - (alpha_bars[1:] / alpha_bars[:-1])
    return torch.clamp(betas, 1e-5, 0.999)


class DiffusionScheduler:
    def __init__(self, num_steps: int, beta_start: float, beta_end: float, device: torch.device, schedule: str = "linear"):
        self.num_steps = num_steps
        if schedule == "cosine":
            betas = cosine_beta_schedule(num_steps)
        else:
            betas = torch.linspace(beta_start, beta_end, num_steps)
        self.betas = betas.to(device)
        self.alphas = 1.0 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def q_sample(self, x0: torch.Tensor, t_index: torch.Tensor, noise: torch.Tensor) -> torch.Tensor:
        alpha_bar = self.alpha_bars[t_index].view(-1, 1, 1)
        return torch.sqrt(alpha_bar) * x0 + torch.sqrt(1.0 - alpha_bar) * noise


class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.detach().clone()

    @torch.no_grad()
    def update(self, model: nn.Module):
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            self.shadow[name].mul_(self.decay).add_(param.detach(), alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, model: nn.Module):
        for name, param in model.named_parameters():
            if not param.requires_grad:
                continue
            param.copy_(self.shadow[name])


model = ConditionalUnet1D(
    action_dim=config.processed_action_dim,
    obs_dim=config.processed_obs_dim,
    base_channels=config.base_channels,
    time_embed_dim=config.time_embed_dim,
    obs_embed_dim=config.obs_embed_dim,
).to(device)

scheduler = DiffusionScheduler(
    config.diffusion_steps,
    config.beta_start,
    config.beta_end,
    device,
    schedule=config.schedule,
)

ema_helper = EMA(model, config.ema_decay) if config.use_ema else None
ema_model = copy.deepcopy(model) if config.use_ema else None
model


ConditionalUnet1D(
  (time_mlp): Sequential(
    (0): SinusoidalPosEmb()
    (1): Linear(in_features=128, out_features=128, bias=True)
    (2): SiLU()
    (3): Linear(in_features=128, out_features=128, bias=True)
  )
  (obs_mlp): Sequential(
    (0): Linear(in_features=23, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (cond_mlp): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): SiLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
  )
  (input_proj): Conv1d(8, 128, kernel_size=(3,), stride=(1,), padding=(1,))
  (down1): ResBlock1D(
    (norm1): GroupNorm(8, 128, eps=1e-05, affine=True)
    (norm2): GroupNorm(8, 128, eps=1e-05, affine=True)
    (conv1): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (cond_proj): Linear(in_features=128, out_features=128, bias=True)
    (skip): Iden

In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)


def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    losses = []
    with torch.no_grad():
        for obs, action_chunk in loader:
            obs = obs.to(device)
            action_chunk = action_chunk.to(device).permute(0, 2, 1)

            t_index = torch.randint(0, scheduler.num_steps, (obs.size(0),), device=device)
            t = t_index.float() / max(scheduler.num_steps - 1, 1)
            noise = torch.randn_like(action_chunk)
            noisy = scheduler.q_sample(action_chunk, t_index, noise)
            pred = model(noisy, t, obs)
            losses.append(F.mse_loss(pred, noise).item())
    return float(np.mean(losses)) if losses else float("nan")


best_val = float("inf")
history = []

for epoch in range(1, config.num_epochs + 1):
    model.train()
    train_losses = []
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{config.num_epochs}")

    for obs, action_chunk in pbar:
        obs = obs.to(device)
        action_chunk = action_chunk.to(device).permute(0, 2, 1)

        t_index = torch.randint(0, scheduler.num_steps, (obs.size(0),), device=device)
        t = t_index.float() / max(scheduler.num_steps - 1, 1)
        noise = torch.randn_like(action_chunk)
        noisy = scheduler.q_sample(action_chunk, t_index, noise)

        pred = model(noisy, t, obs)
        loss = F.mse_loss(pred, noise)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        if ema_helper is not None:
            ema_helper.update(model)
            ema_helper.copy_to(ema_model)

        train_losses.append(loss.item())
        pbar.set_postfix(train_loss=f"{np.mean(train_losses):.6f}")

    eval_model = ema_model if ema_model is not None else model
    val_loss = evaluate(eval_model, val_loader)
    epoch_train_loss = float(np.mean(train_losses))
    history.append((epoch, epoch_train_loss, val_loss))
    print(f"epoch={epoch:03d} train_loss={epoch_train_loss:.6f} val_loss={val_loss:.6f}")

    if val_loss < best_val:
        best_val = val_loss
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "ema_state_dict": ema_model.state_dict() if ema_model is not None else None,
                "config": config.__dict__,
                "best_val": best_val,
            },
            config.checkpoint_path,
        )

best_val


Epoch 1/200: 100%|██████████| 76618/76618 [20:29<00:00, 62.31it/s, train_loss=0.058143]


epoch=001 train_loss=0.058143 val_loss=0.045772


Epoch 2/200:  87%|████████▋ | 66916/76618 [17:50<02:35, 62.49it/s, train_loss=0.045302]


KeyboardInterrupt: 

In [ ]:
@torch.no_grad()
def sample_action_chunk(model: nn.Module, obs_batch: torch.Tensor, num_steps: int | None = None) -> torch.Tensor:
    model.eval()
    num_steps = scheduler.num_steps if num_steps is None else num_steps
    x = torch.randn(obs_batch.size(0), config.processed_action_dim, config.chunk_size, device=device)

    for step in reversed(range(num_steps)):
        t_index = torch.full((obs_batch.size(0),), step, device=device, dtype=torch.long)
        t = t_index.float() / max(num_steps - 1, 1)
        pred_noise = model(x, t, obs_batch)

        alpha = scheduler.alphas[t_index].view(-1, 1, 1)
        alpha_bar = scheduler.alpha_bars[t_index].view(-1, 1, 1)
        beta = scheduler.betas[t_index].view(-1, 1, 1)

        if step > 0:
            noise = torch.randn_like(x)
        else:
            noise = torch.zeros_like(x)

        x = (1.0 / torch.sqrt(alpha)) * (x - ((1.0 - alpha) / torch.sqrt(1.0 - alpha_bar)) * pred_noise) + torch.sqrt(beta) * noise

    return x.permute(0, 2, 1)


# Final validation after training
final_eval_model = ema_model if ema_model is not None else model
final_val_loss = evaluate(final_eval_model, val_loader)
print(f"final_val_loss={final_val_loss:.6f}")


eval_model = ema_model if ema_model is not None else model
batch_obs, batch_target = next(iter(val_loader))
batch_obs = batch_obs[:4].to(device)
pred_chunks = sample_action_chunk(eval_model, batch_obs)

act_mean = torch.from_numpy(dataset.act_mean).to(device)
act_std = torch.from_numpy(dataset.act_std).to(device)
pred_chunks_denorm = pred_chunks * act_std + act_mean

pred_chunks_denorm.shape
